In [1]:
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import time
import pandas as pd
from datetime import datetime

In [2]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

In [3]:
def get_soup(url):
    
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

In [4]:
def parse_page(soup):
    cards = soup.select('div.product-list__item')
    page_data = []
    
    for card in cards:
        brand_tag = card.select_one('.product-list__brand')
        brand = brand_tag.get_text(strip=True) if brand_tag else None
        
        name_tag = card.select_one('.product-list__name')
        name = name_tag.get_text(strip=True) if name_tag else None
        
        price_tag = card.select_one('.product-list__price_new, .product-list__price')
        if price_tag:
            price_text = price_tag.get_text(strip=True)
            price = int(''.join(filter(str.isdigit, price_text)))
        else:
            price = None
    
   
        old_price_tag = card.select_one('.product-list__price_old')
        if old_price_tag:
            old_text = old_price_tag.get_text(strip=True)
            old_price = int(''.join(filter(str.isdigit, old_text)))
        else:
            old_price = None
        
        
        size_tags = card.select('.product-list__size-item')
        sizes = [size.get_text(strip=True) for size in size_tags] if size_tags else None
        
        
        item = {
            'brand': brand,
            'name': name,
            'price': price,
            'old_price': old_price,
            'sizes': sizes,
        }
        page_data.append(item)
    
    return page_data

In [5]:
all_data = []

pages = range(1, 44)

for page in tqdm(pages, desc='Обработка страниц'):
    url = f'https://sport-marafon.ru/catalog/obuv/?page={page}'
    soup = get_soup(url)
    if soup is None:
        break
    all_data.extend(parse_page(soup))
    time.sleep(3)
    page += 1

Обработка страниц: 100%|███████████████████████████████████████████████████████████████| 43/43 [02:40<00:00,  3.74s/it]


In [6]:
df_shoes = pd.DataFrame(all_data)

In [7]:
df_shoes.sample(10)

,brand,name,price,old_price,sizes
147,Saucony,Кроссовки женские Saucony Triumph 23 Gtx Shado...,13188,21980.0,"[37(1/2), 38, 38(1/2), 39, 40, 40(1/2), 42(1/2)]"
890,The North Face,Сандалии женские The North Face Explore Camp T...,11590,NaN,"[38, 38(1/2), 39, 39(1/2), 40, 40(1/2), 41, 41..."
647,Salomon,Кроссовки мужские Salomon Xa Pro 3D V9 India I...,19980,NaN,"[40(2/3), 41(1/3), 42, 42(2/3), 43(1/3), 44, 4..."
770,Mammut,Ботинки женские Mammut Kento Tour High Gtx Dar...,19236,27480.0,"[37(1/3), 38, 38(2/3), 40]"
320,Salomon,Кроссовки мужские Salomon Xa Pro 3D V9 Wide Bl...,19980,NaN,"[42, 42(2/3), 43(1/3), 44, 44(2/3), 45(1/3), 4..."
247,Hoka,Кроссовки женские Hoka Kaha 3 Low Gtx Black/Ci...,18186,25980.0,"[37(1/3), 38, 38(2/3), 39(1/3), 40, 40(2/3)]"
1445,Anta,Кроссовки женские Anta C10 Pro White/Red,35990,NaN,[38]
189,Altra,Кроссовки мужские Altra Escalante 4 White,19290,NaN,"[41, 42, 42(1/2), 43, 44, 44(1/2), 45, 46, 48]"
1400,Lassie,Сандалии детские Lassie Onsala White,1999,3999.0,[29]
1113,Sorel,Ботинки женские Sorel Out N About IV Chillz Wp...,13584,16980.0,"[38(1/2), 41]"


In [8]:
df_shoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1488 entries, 0 to 1487
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   brand      1488 non-null   object 
 1   name       1488 non-null   object 
 2   price      1488 non-null   int64  
 3   old_price  495 non-null    float64
 4   sizes      1488 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 58.2+ KB


In [9]:
today = datetime.now().strftime('%Y-%m-%d')

In [10]:
df_shoes.to_csv(f'sportmarafon_shoes_{today}.csv', index=False, encoding='utf-8-sig')